In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [ ]:
proteomics = pd.read_csv('../../data/ADNI/ADNI_Plasma_withlabel.csv')
img = pd.read_pickle("../../data/ADNI/mri_meta_2D_3class_diagnosislabel.pkl")

In [ ]:
proteomics

In [ ]:
proteomics=proteomics.loc[:, ~(proteomics == '.').all()]

In [ ]:
proteomics

In [ ]:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# proteomics.iloc[:, 9:1025] = scaler.fit_transform(proteomics.iloc[:, 9:1025])

In [ ]:
img

In [ ]:
proteomics['ID_Visit']=None

In [ ]:
proteomics

In [ ]:
proteomics['Visit_Code'].value_counts()

In [ ]:
img['Visit'].value_counts()

In [ ]:
for i in range(len(proteomics)):
    proteomics['ID_Visit'][i]=str(proteomics['RID'][i])+'_'+proteomics['Visit_Code'][i]

In [ ]:
proteomics

In [ ]:
proteomics['label'].value_counts()

In [ ]:
label_set={'NL':0,'MCI':1,'AD':2}

In [ ]:
proteomics['label']=proteomics['label'].map(label_set)

In [ ]:
proteomics

In [ ]:
img

In [ ]:
img['ID_Visit']=None

In [ ]:
for i in range(len(img)):
    img['ID_Visit'][i]=img['RID'][i].split('_')[-1].lstrip("0")+'_'+img['Visit'][i]

In [ ]:
img

In [ ]:
overlap=pd.merge(proteomics,img,on='ID_Visit',how='inner')

In [ ]:
overlap

In [ ]:
overlap.iloc[150]['label_x']

In [ ]:
overlap.iloc[150]['label_y']

In [ ]:
count=0
for i in range(len(overlap)):
    if overlap['label_x'][i] == overlap['label_y'][i]:
        continue
    else:
        count+=1

In [ ]:
count

In [ ]:
overlap

In [ ]:
overlap = overlap[overlap['label_x'] == overlap['label_y']]

In [ ]:
overlap

In [ ]:
overlap = overlap.drop_duplicates(subset=['ID_Visit'])

In [ ]:
overlap

In [ ]:
overlap = overlap.reset_index(drop=True)

In [ ]:
overlap

In [ ]:
overlap=overlap.drop(['Unnamed: 0','RID_x','Visit_Code','label_x','RID_y','Visit'],axis=1)

In [ ]:
overlap

In [ ]:
overlap.index=overlap['ID_Visit']

In [ ]:
overlap

In [ ]:
X= overlap.drop(['ID_Visit','label_y'],axis=1)
y = overlap['label_y']

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)

In [ ]:
X_train

In [ ]:
X_train_plasma = X_train.drop(['img_array'],axis=1)
X_val_plasma = X_val.drop(['img_array'],axis=1)
X_test_plasma = X_test.drop(['img_array'],axis=1)

In [ ]:
img.columns

In [ ]:
img_cols = list(set(img.columns) - set(["ID_Visit","label",'RID','Visit']))
print(len(img.columns))
print(len(img_cols))
X_train_img = X_train[img_cols]
X_val_img = X_val[img_cols]
X_test_img = X_test[img_cols]

In [ ]:
X_train_img.shape

In [ ]:
X_train_plasma.shape

In [ ]:
X_train_plasma.to_csv("../../processed_data/ADNI/overlap/X_train_plasma.csv")
X_val_plasma.to_csv("../../processed_data/ADNI/overlap/X_val_plasma.csv")
X_test_plasma.to_csv("../../processed_data/ADNI/overlap/X_test_plasma.csv")

y_train.to_csv("../../processed_data/ADNI/overlap/y_train.csv")
y_val.to_csv("../../processed_data/ADNI/overlap/y_val.csv")
y_test.to_csv("../../processed_data/ADNI/overlap/y_test.csv")


In [ ]:
pd.DataFrame(X_train_img).to_pickle("../../processed_data/ADNI/overlap/X_train_img.pkl")
pd.DataFrame(X_val_img).to_pickle("../../processed_data/ADNI/overlap/X_val_img.pkl")
pd.DataFrame(X_test_img).to_pickle("../../processed_data/ADNI/overlap/X_test_img.pkl")

In [ ]:
len(X_train_plasma)

In [ ]:
len(X_val_plasma)

In [ ]:
len(X_test_plasma)

In [ ]:
X_train_list=X_train_plasma.index.values.tolist()
X_val_list=X_val_plasma.index.values.tolist()
X_test_list=X_test_plasma.index.values.tolist()

In [ ]:
for i in X_val_list:
    if i in X_test_list:
        print(i)